# Learnible — Chatterbox narration (Colab)

Runs the heavy **Chatterbox TTS** synthesis on a free Colab GPU (your 4 GB card can't).

**Steps**
1. Runtime → **Change runtime type → GPU (T4)**.
2. Run the **install** cell. It prints pip *conflict warnings* about Colab's preinstalled packages — **those are harmless.**
3. **Runtime → Restart session** (essential — so the pinned numpy/torch load cleanly). Do **not** re-run install.
4. Run the **config** cell and **upload `script.json`** (from `web/data/script.json`, produced by `python -m src.webexport --script-only`).
5. Run **load model** → **synth**. It produces `narration.mp3` + `timeline.json` and downloads them.
6. Locally: drop both into `web/data/` and run `python -m src.webexport --merge-audio`.

Optional voice cloning: upload a clean ~10 s WAV and set `AUDIO_PROMPT` to its filename.

In [ ]:
# 1) Install. Chatterbox pins torch==2.6.0, but Colab's preinstalled torchvision
#    is built for a newer torch -> "operator torchvision::nms does not exist".
#    Fix: pin torchvision to the version matching torch 2.6.0 (0.21.0).
#    Also pins numpy==1.26.4. pip will print CONFLICT WARNINGS about Colab's
#    other preinstalled packages (jax, rasterio, shap, cupy, ...) -- harmless,
#    we don't use them.
!pip -q install chatterbox-tts
!pip -q install "numpy==1.26.4" "torch==2.6.0" "torchvision==0.21.0" "torchaudio==2.6.0"
print("\n✅ Installed.  ⚠️  NOW: Runtime → Restart session, then run the cells BELOW this one (do NOT re-run this cell).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24

In [ ]:
# 2) Config + upload script.json
EXAGGERATION = 0.5   # 0.3-0.5 = calm/neutral narration; higher = more expressive
CFG_WEIGHT   = 0.3   # lower = slower, more deliberate pacing (good for audiobooks)
PAUSE_S      = 0.45  # silence between chunks (matches config.CHUNK_PAUSE_MS)
AUDIO_PROMPT = None  # e.g. 'narrator.wav' to clone a voice; None = built-in voice

from google.colab import files
print('Upload script.json:'); _ = files.upload()

Upload script.json:


KeyboardInterrupt: 

In [ ]:
# 3) Load the model (run AFTER restarting the runtime)
import os
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["USE_TORCHVISION"] = "0"   # we don't need torchvision; avoid its lazy-import path

import torch
print('torch:', torch.__version__, '| torchvision import needed: no')
print('CUDA available:', torch.cuda.is_available())   # should be True on a GPU runtime

from chatterbox.tts import ChatterboxTTS
model = ChatterboxTTS.from_pretrained(device='cuda' if torch.cuda.is_available() else 'cpu')
SR = model.sr
print('sample rate:', SR)

In [ ]:
# 4) Synthesise every event -> narration.mp3 + timeline.json
import json, re, subprocess, torch, torchaudio as ta

events = json.load(open('script.json'))

def synth(text):
    # split long paragraphs into sentences for stability, then concat
    parts = [p for p in re.split(r'(?<=[.!?])\s+', text.strip()) if p] or [text]
    kw = dict(exaggeration=EXAGGERATION, cfg_weight=CFG_WEIGHT)
    if AUDIO_PROMPT: kw['audio_prompt_path'] = AUDIO_PROMPT
    wavs = [model.generate(p, **kw) for p in parts]
    return torch.cat([w if w.dim() == 2 else w.unsqueeze(0) for w in wavs], dim=-1)

silence = torch.zeros(1, int(PAUSE_S * SR))
pieces, timeline, t = [], [], 0.0
for n, e in enumerate(events):
    wav = synth(e['text']).cpu()
    dur = wav.shape[-1] / SR
    timeline.append({'i': e['i'], 't0': round(t, 2), 't1': round(t + dur, 2)})
    pieces.append(wav); t += dur
    if e['kind'] == 'content':
        pieces.append(silence); t += PAUSE_S
    if n % 20 == 0: print(f'{n}/{len(events)}  ({t/60:.1f} min)')

ta.save('narration.wav', torch.cat(pieces, dim=-1), SR)
subprocess.run(['ffmpeg','-y','-loglevel','error','-i','narration.wav',
                '-codec:a','libmp3lame','-b:a','64k','narration.mp3'], check=True)
json.dump(timeline, open('timeline.json','w'))
print(f'DONE: {len(events)} events, {t/60:.1f} min')

In [ ]:
# 5) Download -> put both into your local web/data/, then run --merge-audio
from google.colab import files
files.download('narration.mp3')
files.download('timeline.json')